# Data Loading and Setup 

## 1. First Data Inspection

Before performing any transformations, we load the raw dataset and inspect its structure, types, missing values, and unique entries per column.

In [3]:
import pandas as pd

df = pd.read_csv("../data/raw.csv")

# First inspection
print(f"Dataset shape: {df.shape}")
print("\n--- First 5 rows ---")
display(df.head())

print("\n--- Data Types and Missing Values ---")
print(df.info())

print("\n--- Unique values per column ---")
for col in df.columns:
    print(f"{col}: {df[col].unique()[:5]}")  

Dataset shape: (7043, 21)

--- First 5 rows ---


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes



--- Data Types and Missing Values ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  Pap

## 2. Issues Identified & Cleanup Decisions

From our initial inspection, we observe three key technical requirements:
1. **`customerID`**: Unique string identifier per row. It has no predictive power and must be dropped to prevent overfitting.
2. **`TotalCharges`**: Classified as `object` (string) instead of `float64`. Blank spaces (`" "`) exist for new customers (`tenure = 0`), which prevent direct numeric conversion. We will convert them to float and fill missing values with `0`.
3. **Data Partitioning**: Before performing any further transformations or encoding, we must immediately isolate a holdout test set to prevent data leakage. We stratify on Churn to have a equal percentage of yes and no in each dataset (and not false the result).

In [4]:
from sklearn.model_selection import train_test_split

# 1. Drop identifier
if 'customerID' in df.columns:
    df = df.drop(columns=['customerID'])

# 2. Fix TotalCharges
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)

# 3. Holdout split (80% Train / 20% Test)
train_df, test_df = train_test_split(
    df, 
    test_size=0.2, 
    random_state=42, 
    stratify=df['Churn']
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

# Lock and export holdout test set
train_df.to_csv('../data/train_set.csv', index=False)
test_df.to_csv('../data/test_set_holdout.csv', index=False)

print(f"Training set shape: {train_df.shape}")
print(f"Holdout test set shape: {test_df.shape}")

Training set shape: (5634, 20)
Holdout test set shape: (1409, 20)
